In [11]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [ ]:
# read the dataset
dataset = pd.read_csv('D:\\AIWorkspace\\StandardMLProject\\notebooks\\data\\StudentsPerformance.csv')

In [8]:
dataset.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [32]:
# split the dataset into features and target variable
X = dataset.drop('math score', axis=1)
y = dataset['math score']


In [33]:
# lets create the standard scaler and one hot encoder for the categorical features
standard_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder()

X_numerical_features = X.select_dtypes(include=[np.number]).columns
X_categorical_features = X.select_dtypes(include=[object]).columns

C:\Users\Pankaj\AppData\Local\Temp\ipykernel_17688\4146500860.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_categorical_features = X.select_dtypes(include=[object]).columns


In [34]:
X_categorical_features

Index(['gender', 'race/ethnicity', 'parental level of education', 'lunch',
       'test preparation course'],
      dtype='str')

In [35]:
#create a pipeline for preprocessing the data

preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', standard_scaler, X_numerical_features),
        ('encoder', one_hot_encoder, X_categorical_features)
    ]
)

In [36]:
# lets fit the preprocessor on independent features and transform the data
X = preprocessor.fit_transform(X)
X.shape


(1000, 19)

In [37]:
# split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  

In [38]:
X_train.shape

(800, 19)

In [41]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f'Mean Squared Error: {mse}')
    print(f'Root Mean Squared Error: {rmse}')
    print(f'Mean Absolute Error: {mae}')
    print(f'R-squared: {r2}')

In [42]:
models= {
    "linear_model": LinearRegression(),
    "catboost_model": CatBoostRegressor(verbose=0),
    "xgboost_model": XGBRegressor(objective='reg:squarederror', eval_metric='rmse')
}


for i in models:
    print(f"Training {i}...")
    models[i].fit(X_train, y_train)
    print(f"Evaluating {i}...")
    evaluate_model(models[i], X_test, y_test)
    print("\n")

Training linear_model...
Evaluating linear_model...
Mean Squared Error: 29.095169866715487
Root Mean Squared Error: 5.393993869732843
Mean Absolute Error: 4.214763142474851
R-squared: 0.8804332983749565


Training catboost_model...
Evaluating catboost_model...
Mean Squared Error: 36.10365799356841
Root Mean Squared Error: 6.008631956907363
Mean Absolute Error: 4.612531714976557
R-squared: 0.8516318920747058


Training xgboost_model...
Evaluating xgboost_model...
Mean Squared Error: 43.50392150878906
Root Mean Squared Error: 6.595750261250729
Mean Absolute Error: 5.1036295890808105
R-squared: 0.8212205171585083


